<a href="https://colab.research.google.com/github/philippfriberg/swarc4ai/blob/main/LLM_Funktion_call.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google import genai
import os
import json
import inspect

# =========================================
# 🔐 CONFIG
# =========================================
client = genai.Client(api_key="YOUR_KEY")
model_def = 'gemini-3-flash-preview'

CONFIDENCE_THRESHOLD = 0.7
LOW_CONFIDENCE_THRESHOLD = 0.4

# =========================================
# 🧰 TOOL REGISTRY (AUTO DISCOVERY)
# =========================================
TOOLS = {}

def tool(description=""):
    def decorator(func):
        name = func.__name__

        sig = inspect.signature(func)
        params = list(sig.parameters.keys())

        TOOLS[name] = {
            "func": func,
            "description": description,
            "params": params
        }

        return func
    return decorator

# =========================================
# 🛠️ TOOLS
# =========================================
@tool("Gibt das Wetter für eine Stadt zurück")
def get_weather(city):
    return f"In {city} ist es 22°C und sonnig."

@tool("Berechnet mathematische Ausdrücke")
def calculate(expression):
    try:
        return eval(expression)
    except:
        return "Fehler im Ausdruck"

@tool("Sucht in einer Wissensdatenbank")
def search_kb(query):
    db = {
        "microservices": "Architekturstil mit unabhängigen Services",
        "llm": "Large Language Models sind neuronale Netze"
    }
    return db.get(query.lower(), "Keine Ergebnisse")

# =========================================
# 🧠 PROMPT GENERATION
# =========================================
def build_system_prompt():
    prompt = """
Du bist ein Tool-Using-Agent.

Du darfst nur JSON zurückgeben.

Gib zusätzlich eine Confidence (0.0 bis 1.0) an.

Verfügbare Tools:
"""

    for name, meta in TOOLS.items():
        params = ", ".join(meta["params"])
        prompt += f"- {name}({params}): {meta['description']}\n"

    prompt += """
Format:
{
  "tool": "...",
  "confidence": 0.0,
  "arguments": {...}
}

Regeln:
- 0.8-1.0: sehr sicher
- 0.5-0.8: unsicher
- <0.5: wahrscheinlich falsches Tool
"""
    return prompt

# =========================================
# 🤖 GEMINI CALL
# =========================================
def ask_llm(user_input):
    system_prompt = build_system_prompt()

    response = client.models.generate_content(
        model=model_def,
        contents=system_prompt + "\nUser: " + user_input
    )

    return response.text

# =========================================
# 🧩 JSON EXTRACTION
# =========================================
def extract_call(text):
    data = json.loads(text)

    return {
        "tool": data.get("tool"),
        "confidence": data.get("confidence", 0.0),
        "arguments": data.get("arguments", {})
    }

# =========================================
# ⚙️ TOOL EXECUTION
# =========================================
def run_tool(call):
    tool_name = call["tool"]
    args = call["arguments"]

    tool_meta = TOOLS.get(tool_name)

    if not tool_meta:
        return f"Unknown tool: {tool_name}"

    try:
        return tool_meta["func"](**args)
    except Exception as e:
        return f"Execution error: {str(e)}"

# =========================================
# 🧠 CONFIDENCE HANDLING
# =========================================
def handle_call(call):
    confidence = call["confidence"]

    print(f"\nCONFIDENCE: {confidence}")

    if confidence < LOW_CONFIDENCE_THRESHOLD:
        return "Ich bin mir unsicher, welches Tool ich verwenden soll."

    elif confidence < CONFIDENCE_THRESHOLD:
        return f"Unsichere Entscheidung ({confidence}). Bitte präzisiere deine Anfrage."

    else:
        return run_tool(call)

# =========================================
# 💬 FINAL ANSWER GENERATION
# =========================================
def generate_final_answer(user_input, tool_result):
    prompt = f"""
Tool result: {tool_result}
User question: {user_input}
Formuliere eine kurze, natürliche Antwort.
"""

    response = client.models.generate_content(
        model=model_def,
        contents=prompt
    )

    return response.text

# =========================================
# 🔁 MAIN LOOP
# =========================================
def main():
    print("=== LLM Tool Agent mit Confidence ===")

    while True:
        user_input = input("\nUser: ")

        # 1. LLM entscheidet Tool
        raw = ask_llm(user_input)
        print("\nLLM RAW:", raw)

        try:
            call = extract_call(raw)

            # 2. Confidence Handling + Tool Execution
            result = handle_call(call)
            print("\nTOOL RESULT:", result)

            # 3. Final Answer
            final = generate_final_answer(user_input, result)
            print("\nFINAL ANSWER:", final)

        except Exception as e:
            print("\nFehler beim Parsen:", e)

# =========================================
# ▶️ START
# =========================================
if __name__ == "__main__":
    main()